# Exercise 3 Digitization and Data Analytics - Machine Learning

## Supervised Learning - Bayes Classifier

Content

- Fundamental concepts in interaction with Apache Spark
- Usage of basic algorithms (Bayes Classifier) from the machine library MLLib within Spark

In [1]:
!pip install pyspark

  Using cached pyspark-4.1.2.tar.gz (455.4 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached py4j-0.10.9.9-py2.py3-none-any.whl.metadata (1.3 kB)
Using cached py4j-0.10.9.9-py2.py3-none-any.whl (203 kB)
  Created wheel for pyspark: filename=pyspark-4.1.2-py2.py3-none-any.whl size=456079515 sha256=bc1945e3dd57d1664e5c1efcf202e78fcabce1ae183934151768184e2b424006
  Stored in directory: /Users/krishnakumarm/Library/Caches/pip/wheels/e6/9c/35/b08622081a09dc48b9467b570ae170519430915aa3c8d27cf9
Successfully built pyspark
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyspark]m1/2 [pyspark]


In [2]:
%matplotlib inline
import os


# Adapt the path to point to your iris.scale and iris.csv
scaledIrisDataPath = "iris.scale"
irisDataPath = "iris.csv"


import platform
print('the python version is: ' + platform.python_version())
import pyspark
from pyspark import SparkContext

ModuleNotFoundError: No module named 'matplotlib'

### Load the Bayes Classifier library:

In [ ]:
# $example on$
from pyspark.ml import Pipeline
from pyspark.ml.classification import NaiveBayes, DecisionTreeClassifier
from pyspark.ml.feature import StringIndexer, VectorIndexer, VectorAssembler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
# $example off$
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType, StructType, StructField, StringType


In [ ]:
spark = SparkSession.builder.appName("NaiveBayesClassificationExample").getOrCreate()

Load your data and preprocess it as already done in previous examples. 

In [ ]:
# generate DataFrame, use DataFrameReader class to parse csv input
dataframe=spark.read.csv(irisDataPath,
                    schema=StructType([StructField("sepal_length",DoubleType(),True),
                                       StructField("sepal_width",DoubleType(),True),
                                       StructField("petal_length",DoubleType(),True),
                                       StructField("petal_width",DoubleType(),True),
                                       StructField("spezies",StringType(),True)]))
#define a corresponding vector assembler
assembler = VectorAssembler(inputCols=["sepal_length",
                                 "sepal_width",
                                 "petal_length",
                                 "petal_width"],
                      outputCol="features")
dataset = assembler.transform(dataframe)
dataset.printSchema()
dataset.show(15, False)

#### Deal with Categorical Labels and Variables

#### Exercise:

Define a transformation, that adds indices to the labels in your data (use StringIndexer), and fit it to your data.
Why it is necessary to have a "fit" and a "transform" step to obtain the labels in the indexed manner?

In [ ]:
# Convert target into numerical categories
labelIndexer = StringIndexer(inputCol="spezies", outputCol="indexedLabel")

In [ ]:
labelIndexerModel = labelIndexer.fit(dataframe)
dataframeLabelIndexed = labelIndexerModel.transform(dataframe)
dataframeLabelIndexed.printSchema()
dataframeLabelIndexed.show(15, False)

#### Training and Test Data Sets

#### Exercise:
Split your data into training and test data.

In [ ]:
# Split dataset randomly into Training and Test sets (30% held out for testing). Set seed for reproducibility
(trainingData, testData) = dataframe.randomSplit([0.7, 0.3], seed = 100)

print(trainingData.count())
print(testData.count())

#### Exercise:
Use the above defined label indexer to show the indexed test- and training data.

In [ ]:
testDataLabelIndexed = labelIndexerModel.transform(testData)

#### Train a Naive Bayes Classifier

#### Exercise:
Define a Naive Bayes classifier for the iris data set. Have a look at the documentation of  NaiveBayes() and its parameters:  
1. https://spark.apache.org/docs/2.2.0/api/python/_modules/pyspark/ml/classification.html  
2. https://spark.apache.org/docs/2.2.0/api/python/pyspark.ml.html?highlight=naivebayes#pyspark.ml.classification.NaiveBayes).

Hint: Pay attention to the labels within your data, you use for the classifier.

In [ ]:
nb = ...

#### Exercise:
1. Read the documentation about the pipeline-component (see https://spark.apache.org/docs/latest/ml-pipeline.html#pipeline-components).
2. Define a pipeline, describing the process of indexing and transforming your data, and then training the naive bayes classifier.
3. Train the model.

In [ ]:
pipeline = Pipeline(stages=[...])

In [ ]:
# Run stages in pipeline and train model
model = ...

#### Use the trained model for predictions

In [ ]:
# Make predictions.
predictions = ...

In [ ]:
# Select example rows to display.
predictions.select("prediction", "indexedLabel", "features").show(15)

In [ ]:
# Show the same with the original labels
converter = IndexToString(inputCol="indexedLabel", outputCol="originalLabel")
convertedPrediction = converter.transform(predictions)
convertedPrediction.select("prediction", "originalLabel", "features").show(15)

#### Evaluate your results

In [ ]:
# Select (prediction, true label) and compute test error
evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print("Test Error = %g " % (1.0 - accuracy))

#### Exercise:

1. Change the partition of test data, rerun and compare the test errors.
How many training data are necessary to keep the error in a reasonable intervall?
2. Change the metric and observe F1, precision, and recall. (https://spark.apache.org/docs/latest/api/java/org/apache/spark/ml/evaluation/MulticlassClassificationEvaluator.html)

In [ ]:
spark.stop()